# Test Your Trained Model: CIFAR-10 Prediction Demo

Loads a trained model (baseline CNN or the fine-tuned ResNet50) and predicts classes with confidence scores:

1. On random CIFAR-10 test images (true vs predicted label)
2. On **your own uploaded images**

**Prerequisites:** run `01-cifar10-training.ipynb` first (or `02-transfer-learning.ipynb`).
Set `INPUT_SIZE` below to match the model you trained: `32` for the baseline CNN, `224` for ResNet50.

In [ ]:
# Install extra packages (TensorFlow is pre-installed in Colab)
%pip install -q numpy matplotlib tensorflow scikit-learn

In [ ]:
import os
import io
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from google.colab import files

CLASS_NAMES = ["airplane", "automobile", "bird", "cat", "deer",
               "dog", "frog", "horse", "ship", "truck"]

# ---- Match this to your model ----
INPUT_SIZE = 32        # 32 for baseline CNN, 224 for the fine-tuned ResNet50
MODEL_FILE = "/content/models/cifar10_cnn_final.keras"
# MODEL_FILE = "/content/models/cifar10_resnet50_finetuned.keras"  # uncomment for transfer model

print("TensorFlow:", tf.__version__)

## 1) Load a Trained Model

If the model file was already saved in this session, it is loaded directly.
Otherwise, use the file picker to upload your `.keras` model (from Drive or your computer).

In [ ]:
if os.path.exists(MODEL_FILE):
    print("Loading", MODEL_FILE)
    model = keras.models.load_model(MODEL_FILE)
else:
    print("Model not found in session. Upload it now (choose your .keras file):")
    uploaded = files.upload()
    fname = list(uploaded.keys())[0]
    model = keras.models.load_model(fname)

print("Model loaded. Input shape:", model.input_shape)

## 2) Predict on CIFAR-10 Test Images

In [ ]:
def preprocess_batch(images, size=INPUT_SIZE):
    ims = np.array([tf.image.resize(img, (size, size)).numpy() for img in images])
    ims = ims.astype("float32") / 255.0
    if size == 224:               # ResNet50 expects [-1, 1]
        ims = ims * 2.0 - 1.0
    return ims

def predict_top_k(model, image, k=3):
    # image: single (H, W, 3) image already preprocessed
    probs = model.predict(image[None, ...], verbose=0)[0]
    top = np.argsort(probs)[::-1][:k]
    return [(CLASS_NAMES[i], float(probs[i])) for i in top]

# Load CIFAR-10 test set to demo on real, labeled images
(_, _), (x_test, y_test) = keras.datasets.cifar10.load_data()

rng = np.random.RandomState(7)
sample_idx = rng.choice(len(x_test), size=6, replace=False)
samples = preprocess_batch(x_test[sample_idx])

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = np.atleast_1d(axes).ravel()
for ax, i, s in zip(axes, sample_idx, samples):
    true_label = CLASS_NAMES[int(y_test[i])]
    top = predict_top_k(model, s)
    pred_label, conf = top[0]
    ax.imshow(x_test[i].astype("uint8"))
    ax.set_title(f"True: {true_label}\nPred: {pred_label} ({conf:.2%})", fontsize=9,
                 color="green" if pred_label == true_label else "red")
    ax.axis("off")
plt.suptitle(f"Model Predictions (top-1 confidence)", y=1.02)
plt.tight_layout()
plt.show()

## 3) Predict on Your Own Uploaded Images

Upload one or more images (JPG/PNG). They are resized to the model's input size and
shown with the predicted class and top-3 confidence scores.

In [ ]:
from PIL import Image

uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

def load_image(data_bytes):
    img = Image.open(io.BytesIO(data_bytes)).convert("RGB")
    img = img.resize((INPUT_SIZE, INPUT_SIZE), Image.LANCZOS)
    arr = np.asarray(img, dtype="float32") / 255.0
    if INPUT_SIZE == 224:
        arr = arr * 2.0 - 1.0
    return arr

cols = 3
rows = int(np.ceil(len(uploaded) / cols)) or 1
fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 3.5))
axes = np.atleast_1d(axes).ravel()

for ax, (name, data) in zip(axes, uploaded.items()):
    img_arr = load_image(data)
    top = predict_top_k(model, img_arr)
    ax.imshow(np.clip(img_arr if INPUT_SIZE != 224 else (img_arr + 1) / 2, 0, 1))
    title = f"{name}\n"
    title += "\n".join(f"{lbl}  {c:.2%}" for lbl, c in top)
    ax.set_title(title, fontsize=8)
    ax.axis("off")

for ax in axes[len(uploaded):]:
    ax.axis("off")
plt.suptitle("Custom Image Predictions (top-3)", y=1.02)
plt.tight_layout()
plt.show()

## 4) Confidence Details

Run this to print the full probability vector for any uploaded image above.

In [ ]:
name = list(uploaded.keys())[0] if uploaded else None
if name is not None:
    img_arr = load_image(uploaded[name])
    probs = model.predict(img_arr[None, ...], verbose=0)[0]
    top = np.argsort(probs)[::-1]
    print(f"Image: {name}\n")
    for i in top[:5]:
        print(f"{CLASS_NAMES[i]:10s} {probs[i]:.2%}")
    print("\nAll class probabilities:", dict(zip(CLASS_NAMES, probs.round(3))))